# Computational Gastronomy - Network Analysis

**Student:** Aurora Felisari  
**ID:** 397867  
**Course:** Artificial Intelligence Lab  
**Academic Year:** 2025/2026

## Project Overview

This notebook presents a dual-level network analysis of culinary structures:

### 1. Topological Analysis of Recipes
Quantitative comparison between traditional and avant-garde cuisine through:
- Centrality Measures (Degree, Betweenness, Closeness, Eigenvector)
- Community Detection (Louvain algorithm)
- Small-World Metrics (Clustering coefficient, path length, sigma)

### 2. Chemical Analysis (Flavor Network)
Scientific validation using Ahn et al. (2011) shared aromatic compounds:
- Bipartite Graph (Ingredient-Compound relationships)
- Flavor Pairing Hypothesis validation
- Molecular Profile Analysis (Jaccard similarity)

### 3. Interactive Bridge Discovery
Comparative analysis of ingredient connectivity through centrality-based methods.

## Configuration and Dependencies

In [ ]:
# Install required packages
import sys
!{sys.executable} -m pip install -q pandas networkx matplotlib scipy seaborn numpy python-louvain
print("Packages installed successfully.")

In [ ]:
# Import libraries
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from collections import defaultdict, Counter
from scipy import stats
from scipy.stats import mannwhitneyu, ks_2samp
import community as community_louvain
import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully.")
print(f"NetworkX version: {nx.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
def load_r_matrix(filepath):
    """
    Load adjacency matrix from R output file.
    """
    with open(filepath, 'r') as f:
        content = f.read()
    
    tokens = content.split()
    numbers = [float(t) for t in tokens if t.replace('.','').replace('-','').isdigit()]
    
    size = int(len(numbers)**0.5)
    if size * size != len(numbers):
        numbers = numbers[:size*size]
    
    return np.array(numbers).reshape(size, size)


def create_recipe_graph(matrix, threshold=0.0, make_symmetric=True):
    """
    Create NetworkX graph from adjacency matrix.
    """
    if make_symmetric:
        matrix = np.maximum(matrix, matrix.T)
    
    G = nx.Graph()
    rows, cols = matrix.shape
    G.add_nodes_from(range(rows))
    
    for i in range(rows):
        for j in range(i + 1, cols):
            if matrix[i, j] > threshold:
                G.add_edge(i, j, weight=matrix[i, j])
    
    return G


def print_graph_summary(G, name="Network"):
    """
    Print basic graph statistics.
    """
    print(f"\n{'='*60}")
    print(f"{name.upper()} - BASIC STATISTICS")
    print(f"{'='*60}")
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")
    print(f"Density: {nx.density(G):.4f}")
    print(f"Connected: {nx.is_connected(G)}")
    
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        print(f"Connected Components: {len(components)}")
        print(f"Largest Component Size: {len(max(components, key=len))} nodes")
    
    degrees = [d for n, d in G.degree()]
    print(f"Average Degree: {np.mean(degrees):.2f}")
    print(f"Degree Range: [{min(degrees)}, {max(degrees)}]")
    print(f"{'='*60}")


print("Utility functions defined.")

## Recipe Network Data Loading

In [ ]:
# File paths
PATH_CTRAD = 'data/Ctrad_mat_ricette_substitution_output.txt'
PATH_ROCA = 'data/Roca_mat_ricette_output.txt'

# Load matrices
print("Loading adjacency matrices...")
mat_ctrad = load_r_matrix(PATH_CTRAD)
mat_roca = load_r_matrix(PATH_ROCA)

print(f"\nTraditional matrix shape: {mat_ctrad.shape}")
print(f"Roca matrix shape: {mat_roca.shape}")

# Create graphs
print("\nCreating network graphs...")
G_ctrad = create_recipe_graph(mat_ctrad, threshold=0.0, make_symmetric=True)
G_roca = create_recipe_graph(mat_roca, threshold=0.0, make_symmetric=True)

# Print summaries
print_graph_summary(G_ctrad, "Traditional Cuisine")
print_graph_summary(G_roca, "Avant-Garde Cuisine (Roca)")

# Size bias warning
size_ratio = len(G_ctrad) / len(G_roca) if len(G_roca) > 0 else 0
print(f"\nNOTE: Traditional network is {size_ratio:.2f}x larger than Roca network.")
print("Bootstrap sampling will be applied for fair comparison.")

## Flavor Network Data Loading (Ahn et al. 2011)

In [ ]:
# File paths
PATH_INGR_INFO = 'data/ingr_info.tsv'
PATH_INGR_COMP = 'data/ingr_comp.tsv'

print("Loading flavor network data (Ahn et al. 2011)...")

# Load ingredient information
ingr_info = pd.read_csv(PATH_INGR_INFO, sep='\t', comment='#', 
                        names=['id', 'name', 'category'])

# Load ingredient-compound relationships
ingr_comp = pd.read_csv(PATH_INGR_COMP, sep='\t', comment='#',
                        names=['ingredient_id', 'compound_id'])

print(f"\nIngredients: {len(ingr_info)}")
print(f"Ingredient-Compound pairs: {len(ingr_comp)}")
print(f"Unique compounds: {ingr_comp['compound_id'].nunique()}")

# Create mapping dictionaries
id_to_name = pd.Series(ingr_info['name'].values, index=ingr_info['id']).to_dict()
id_to_category = pd.Series(ingr_info['category'].values, index=ingr_info['id']).to_dict()
name_to_id = pd.Series(ingr_info['id'].values, index=ingr_info['name']).to_dict()

# Create ingredient -> compounds mapping
ingredient_compounds = defaultdict(set)
for _, row in ingr_comp.iterrows():
    ingredient_compounds[row['ingredient_id']].add(row['compound_id'])

print(f"\nMapping dictionaries created.")

In [ ]:
print("Building bipartite graph (Ingredients - Compounds)...\n")

# Create bipartite graph
B = nx.Graph()

# Add ingredient nodes (bipartite set 0)
known_ingredients = set()
for _, row in ingr_info.iterrows():
    node_id = f"i_{row['id']}"
    B.add_node(node_id, bipartite=0, name=row['name'], category=row['category'])
    known_ingredients.add(node_id)

# Add compound nodes (bipartite set 1) and edges
compounds_seen = set()
edges_to_add = []

for _, row in ingr_comp.iterrows():
    ing_node = f"i_{row['ingredient_id']}"
    comp_node = f"c_{row['compound_id']}"
    
    if ing_node in known_ingredients:
        if comp_node not in compounds_seen:
            B.add_node(comp_node, bipartite=1)
            compounds_seen.add(comp_node)
        edges_to_add.append((ing_node, comp_node))

B.add_edges_from(edges_to_add)

print(f"Bipartite graph created:")
print(f"  - Ingredient nodes: {len(known_ingredients)}")
print(f"  - Compound nodes: {len(compounds_seen)}")
print(f"  - Total edges: {B.number_of_edges()}")

# Project to ingredient-ingredient network
print("\nProjecting to Flavor Network (Ingredient - Ingredient)...")
print("NOTE: This operation may take 30-60 seconds.\n")

ingredient_nodes = [n for n, d in B.nodes(data=True) if d.get('bipartite') == 0]
G_flavor = nx.bipartite.weighted_projected_graph(B, ingredient_nodes)

print(f"Flavor Network created:")
print(f"  - Ingredients: {G_flavor.number_of_nodes()}")
print(f"  - Shared compound connections: {G_flavor.number_of_edges()}")
print(f"  - Density: {nx.density(G_flavor):.4f}")
print(f"  - Average shared compounds: {np.mean([d['weight'] for u, v, d in G_flavor.edges(data=True)]):.2f}")

## Bootstrap Sampling for Size-Corrected Comparison

To ensure fair comparison, 100 random subgraphs are sampled from the traditional network matching the size of the Roca network.

In [ ]:
def calculate_advanced_metrics(G, name="Network", calculate_centrality=False):
    """
    Calculate comprehensive network metrics.
    """
    n = G.number_of_nodes()
    m = G.number_of_edges()
    
    if n <= 1:
        return None
    
    metrics = {'name': name, 'nodes': n, 'edges': m}
    
    # Basic metrics
    metrics['density'] = nx.density(G)
    metrics['avg_clustering'] = nx.average_clustering(G)
    
    # Community detection (Louvain)
    try:
        partition = community_louvain.best_partition(G)
        communities = defaultdict(list)
        for node, comm_id in partition.items():
            communities[comm_id].append(node)
        
        metrics['n_communities'] = len(communities)
        metrics['modularity'] = community_louvain.modularity(partition, G)
    except:
        metrics['n_communities'] = 0
        metrics['modularity'] = 0
    
    # Path-based metrics
    if nx.is_connected(G):
        metrics['avg_path_length'] = nx.average_shortest_path_length(G)
        metrics['diameter'] = nx.diameter(G)
        G_analysis = G
    else:
        largest_cc = max(nx.connected_components(G), key=len)
        G_analysis = G.subgraph(largest_cc)
        metrics['avg_path_length'] = nx.average_shortest_path_length(G_analysis) if len(G_analysis) > 1 else 0
        metrics['diameter'] = nx.diameter(G_analysis) if len(G_analysis) > 1 else 0
    
    # Small-world coefficient (sigma)
    G_random = nx.erdos_renyi_graph(n, metrics['density'], seed=42)
    C_random = nx.average_clustering(G_random)
    try:
        L_random = nx.average_shortest_path_length(G_random)
    except:
        L_random = metrics['avg_path_length']
    
    if C_random > 0 and L_random > 0 and metrics['avg_path_length'] > 0:
        metrics['sigma'] = (metrics['avg_clustering'] / C_random) / (metrics['avg_path_length'] / L_random)
    else:
        metrics['sigma'] = 0
    
    # Degree statistics
    degrees = [d for n, d in G.degree()]
    metrics['avg_degree'] = np.mean(degrees)
    metrics['std_degree'] = np.std(degrees)
    metrics['min_degree'] = np.min(degrees)
    metrics['max_degree'] = np.max(degrees)
    
    return metrics


print("Metrics function defined.")

In [ ]:
print("Bootstrap Sampling - 100 iterations\n")

N_BOOTSTRAP = 100
target_size = len(G_roca)
bootstrap_results = []
np.random.seed(42)

print(f"Sampling {N_BOOTSTRAP} subgraphs of size {target_size} from Traditional network...\n")

for i in range(N_BOOTSTRAP):
    sampled_nodes = np.random.choice(list(G_ctrad.nodes()), size=target_size, replace=False)
    G_sampled = G_ctrad.subgraph(sampled_nodes)
    
    metrics = calculate_advanced_metrics(G_sampled, f"Sample_{i}", calculate_centrality=False)
    
    if metrics:
        bootstrap_results.append({
            'density': metrics['density'],
            'modularity': metrics['modularity'],
            'clustering': metrics['avg_clustering'],
            'path_length': metrics['avg_path_length'],
            'sigma': metrics['sigma'],
            'diameter': metrics['diameter'],
            'avg_degree': metrics['avg_degree'],
            'n_communities': metrics['n_communities']
        })
    
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{N_BOOTSTRAP}")

bootstrap_df = pd.DataFrame(bootstrap_results)
print(f"\nBootstrap sampling complete. {len(bootstrap_df)} samples collected.\n")

# Calculate Roca metrics
metrics_roca = calculate_advanced_metrics(G_roca, "Roca", calculate_centrality=False)

# Display statistics
print("="*80)
print("BOOTSTRAP STATISTICS (Traditional network sampled to match Roca size)")
print("="*80)
print(bootstrap_df.describe().round(4))
print("\n" + "="*80)

## Network Topology Comparison

In [43]:
# Roca metrics for comparison
roca_metrics = {
    'density': metrics_roca['density'],
    'modularity': metrics_roca['modularity'],
    'clustering': metrics_roca['avg_clustering'],
    'path_length': metrics_roca['avg_path_length'],
    'sigma': metrics_roca['sigma'],
    'diameter': metrics_roca['diameter'],
    'avg_degree': metrics_roca['avg_degree'],
    'n_communities': metrics_roca['n_communities']
}

# Create comparison table
comparison_bootstrap = pd.DataFrame({
    'Metric': ['Density', 'Modularity', 'Clustering', 'Path Length', 'Sigma (σ)', 'Diameter', 'Avg Degree', 'N Communities'],
    'Traditional (mean)': [
        f"{bootstrap_df['density'].mean():.4f}",
        f"{bootstrap_df['modularity'].mean():.4f}",
        f"{bootstrap_df['clustering'].mean():.4f}",
        f"{bootstrap_df['path_length'].mean():.4f}",
        f"{bootstrap_df['sigma'].mean():.4f}",
        f"{bootstrap_df['diameter'].mean():.2f}",
        f"{bootstrap_df['avg_degree'].mean():.2f}",
        f"{bootstrap_df['n_communities'].mean():.2f}"
    ],
    'Traditional (std)': [
        f"{bootstrap_df['density'].std():.4f}",
        f"{bootstrap_df['modularity'].std():.4f}",
        f"{bootstrap_df['clustering'].std():.4f}",
        f"{bootstrap_df['path_length'].std():.4f}",
        f"{bootstrap_df['sigma'].std():.4f}",
        f"{bootstrap_df['diameter'].std():.2f}",
        f"{bootstrap_df['avg_degree'].std():.2f}",
        f"{bootstrap_df['n_communities'].std():.2f}"
    ],
    'Roca': [
        f"{roca_metrics['density']:.4f}",
        f"{roca_metrics['modularity']:.4f}",
        f"{roca_metrics['clustering']:.4f}",
        f"{roca_metrics['path_length']:.4f}",
        f"{roca_metrics['sigma']:.4f}",
        f"{roca_metrics['diameter']}",
        f"{roca_metrics['avg_degree']:.2f}",
        f"{roca_metrics['n_communities']}"
    ]
})

print("\n" + "="*80)
print("SIZE-CORRECTED COMPARISON (Bootstrap vs Roca)")
print("="*80)
print(comparison_bootstrap.to_string(index=False))
print("\n" + "="*80)


SIZE-CORRECTED COMPARISON (Bootstrap vs Roca)
       Metric Traditional (mean) Traditional (std)   Roca
      Density             0.9731            0.0134 0.7625
   Modularity             0.0356            0.0036 0.0587
   Clustering             0.9773            0.0100 0.8688
  Path Length             1.0269            0.0134 1.2375
    Sigma (σ)             1.0077            0.0039 1.1499
     Diameter               2.00              0.00      2
   Avg Degree              54.49              0.75  42.70
N Communities               4.17              0.64      4



## Statistical Significance Analysis

In [44]:
# Perform Mann-Whitney U tests
print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTS (Mann-Whitney U)")
print("="*80)
print("Null Hypothesis: Roca and Traditional (bootstrap) come from same distribution\n")

metrics_to_test = ['density', 'modularity', 'clustering', 'path_length', 'sigma', 'avg_degree']

for metric in metrics_to_test:
    bootstrap_values = bootstrap_df[metric].values
    roca_value = roca_metrics[metric]
    
    # Create array for Roca (repeated value for comparison)
    roca_array = np.array([roca_value] * len(bootstrap_values))
    
    # Perform test
    statistic, p_value = mannwhitneyu(bootstrap_values, roca_array, alternative='two-sided')
    
    # Determine significance
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    else:
        sig = "ns"
    
    print(f"{metric.upper():20s}: p = {p_value:.6f} {sig}")

print("\n*** p < 0.001, ** p < 0.01, * p < 0.05, ns = not significant")
print("="*80)


STATISTICAL SIGNIFICANCE TESTS (Mann-Whitney U)
Null Hypothesis: Roca and Traditional (bootstrap) come from same distribution

DENSITY             : p = 0.000000 ***
MODULARITY          : p = 0.000000 ***
CLUSTERING          : p = 0.000000 ***
PATH_LENGTH         : p = 0.000000 ***
SIGMA               : p = 0.000000 ***
AVG_DEGREE          : p = 0.000000 ***

*** p < 0.001, ** p < 0.01, * p < 0.05, ns = not significant


## Ingredient Pair Analysis Functions

In [ ]:
def search_ingredients(search_term, max_results=20):
    """
    Search for ingredients by partial name match.
    """
    term = search_term.lower().strip()
    matches = []
    for name in name_to_id.keys():
        if term in str(name).lower():
            matches.append(str(name))
    matches.sort(key=len)
    return matches[:max_results]


def analyze_ingredient_pair(name1, name2):
    """
    Comprehensive analysis of two ingredients' connection.
    """
    if name1 not in name_to_id or name2 not in name_to_id:
        return {'error': 'One or both ingredients not found in database.'}
    
    id1 = name_to_id[name1]
    id2 = name_to_id[name2]
    
    # Basic info
    cat1 = id_to_category.get(id1, 'unknown')
    cat2 = id_to_category.get(id2, 'unknown')
    
    # Compound analysis
    comp1 = ingredient_compounds.get(id1, set())
    comp2 = ingredient_compounds.get(id2, set())
    shared_compounds = comp1 & comp2
    
    # Jaccard similarity
    union = comp1 | comp2
    jaccard_sim = len(shared_compounds) / len(union) if len(union) > 0 else 0
    
    # Flavor network connection
    node1 = f"i_{id1}"
    node2 = f"i_{id2}"
    connected_flavor = G_flavor.has_edge(node1, node2)
    weight_flavor = G_flavor[node1][node2]['weight'] if connected_flavor else 0
    
    # Path analysis
    try:
        if node1 in G_flavor and node2 in G_flavor:
            if nx.has_path(G_flavor, node1, node2):
                shortest_path = nx.shortest_path(G_flavor, node1, node2)
                path_length = len(shortest_path) - 1
                path_names = [id_to_name[int(n.split('_')[1])] for n in shortest_path]
            else:
                shortest_path = []
                path_length = float('inf')
                path_names = []
        else:
            shortest_path = []
            path_length = float('inf')
            path_names = []
    except:
        shortest_path = []
        path_length = float('inf')
        path_names = []
    
    result = {
        'name1': name1, 'name2': name2,
        'id1': id1, 'id2': id2,
        'category1': cat1, 'category2': cat2,
        'n_compounds1': len(comp1), 'n_compounds2': len(comp2),
        'shared_compounds': list(shared_compounds),
        'n_shared': len(shared_compounds),
        'jaccard_similarity': jaccard_sim,
        'connected_flavor': connected_flavor,
        'weight_flavor': int(weight_flavor),
        'path_length': path_length,
        'path_names': path_names,
        'shortest_path': shortest_path
    }
    
    return result


print("Ingredient analysis functions defined.")

## Bridge Discovery Analysis Functions

In [ ]:
def find_bridge_nodes(name1, name2, method='betweenness', top_n=5):
    """
    Find best bridge ingredients between two ingredients.
    
    Parameters:
    -----------
    name1, name2 : str
        Ingredient names
    method : str
        'betweenness' or 'optimal_path'
    top_n : int
        Number of top bridges to return
    """
    if name1 not in name_to_id or name2 not in name_to_id:
        return []
    
    id1 = name_to_id[name1]
    id2 = name_to_id[name2]
    node1 = f"i_{id1}"
    node2 = f"i_{id2}"
    
    if node1 not in G_flavor or node2 not in G_flavor:
        return []
    
    if method == 'betweenness':
        # Optimized betweenness approach
        neighbors1 = set(G_flavor.neighbors(node1))
        neighbors2 = set(G_flavor.neighbors(node2))
        
        common_neighbors = neighbors1 & neighbors2
        if not common_neighbors:
            extended = common_neighbors.copy()
            for n in list(neighbors1)[:50]:
                extended.update(list(G_flavor.neighbors(n))[:20])
            for n in list(neighbors2)[:50]:
                extended.update(list(G_flavor.neighbors(n))[:20])
            common_neighbors = extended & neighbors1 & neighbors2
        
        common_neighbors.discard(node1)
        common_neighbors.discard(node2)
        
        if not common_neighbors:
            return []
        
        subgraph_nodes = {node1, node2} | set(list(common_neighbors)[:100])
        G_sub = G_flavor.subgraph(subgraph_nodes)
        
        between = nx.betweenness_centrality(G_sub, weight='weight')
        
        bridges = [(id_to_name[int(n.split('_')[1])], score)
                   for n, score in between.items()
                   if n != node1 and n != node2 and score > 0]
        bridges.sort(key=lambda x: x[1], reverse=True)
        return bridges[:top_n]
    
    elif method == 'optimal_path':
        try:
            if not nx.has_path(G_flavor, node1, node2):
                return []
            
            shortest_path = nx.shortest_path(G_flavor, node1, node2)
            if len(shortest_path) <= 2:
                return []
            
            node_scores = {}
            
            # Nodes on shortest path get base score
            for node in shortest_path[1:-1]:
                node_scores[node] = 10.0
            
            # Add score based on edge weights
            for node in shortest_path[1:-1]:
                score = 0.0
                if G_flavor.has_edge(node, node1):
                    score += G_flavor[node][node1]['weight'] / 100.0
                if G_flavor.has_edge(node, node2):
                    score += G_flavor[node][node2]['weight'] / 100.0
                node_scores[node] = node_scores.get(node, 0) + score
            
            # Check common neighbors
            neighbors1 = set(G_flavor.neighbors(node1))
            neighbors2 = set(G_flavor.neighbors(node2))
            common = neighbors1 & neighbors2
            for node in common:
                if node not in node_scores:
                    w1 = G_flavor[node][node1]['weight']
                    w2 = G_flavor[node][node2]['weight']
                    node_scores[node] = (w1 + w2) / 100.0
            
            bridges = [(id_to_name[int(n.split('_')[1])], score)
                       for n, score in node_scores.items()]
            bridges.sort(key=lambda x: x[1], reverse=True)
            return bridges[:top_n]
        except Exception as e:
            print(f"Error in optimal_path: {e}")
            return []
    
    return []


print("Bridge discovery functions defined.")

## Case Study 1: Beef and Wine

In [47]:
print("="*80)
print("EXAMPLE 1: BEEF & WINE")
print("="*80)

# Analyze pair
result = analyze_ingredient_pair('beef', 'wine')

if 'error' not in result:
    print(f"\nAnalyzing: {result['name1'].upper()} <-> {result['name2'].upper()}\n")
    print(f"Basic Info:")
    print(f"  {result['name1']}: {result['category1']}, {result['n_compounds1']} compounds")
    print(f"  {result['name2']}: {result['category2']}, {result['n_compounds2']} compounds")
    print(f"\nChemical Similarity:")
    print(f"  Shared compounds: {result['n_shared']}")
    print(f"  Jaccard similarity: {result['jaccard_similarity']:.4f}")
    print(f"\nFlavor Network:")
    print(f"  Direct connection: {'YES' if result['connected_flavor'] else 'NO'}")
    if result['connected_flavor']:
        print(f"  Shared compounds (edge weight): {result['weight_flavor']}")
    print(f"  Path length: {result['path_length']}")
    if result['path_length'] < float('inf') and result['path_length'] <= 5:
        print(f"  Path: {' -> '.join(result['path_names'])}")
else:
    print(result['error'])

# Find bridges
print("\n" + "-"*80)
print("BRIDGE DISCOVERY")
print("-"*80)

print("\nMethod 1: Betweenness Centrality")
bridges_between = find_bridge_nodes('beef', 'wine', method='betweenness', top_n=10)
if bridges_between:
    for i, (name, score) in enumerate(bridges_between, 1):
        print(f"  {i}. {name:30s} (score: {score:.4f})")
else:
    print("  No bridges found")

print("\nMethod 2: Optimal Path Centrality")
bridges_path = find_bridge_nodes('beef', 'wine', method='optimal_path', top_n=10)
if bridges_path:
    for i, (name, score) in enumerate(bridges_path, 1):
        print(f"  {i}. {name:30s} (score: {score:.4f})")
else:
    print("  No bridges found")

print("\n" + "="*80)

EXAMPLE 1: BEEF & WINE

Analyzing: BEEF <-> WINE

Basic Info:
  beef: meat, 199 compounds
  wine: alcoholic beverage, 128 compounds

Chemical Similarity:
  Shared compounds: 53
  Jaccard similarity: 0.1934

Flavor Network:
  Direct connection: YES
  Shared compounds (edge weight): 53
  Path length: 1
  Path: beef -> wine

--------------------------------------------------------------------------------
BRIDGE DISCOVERY
--------------------------------------------------------------------------------

Method 1: Betweenness Centrality
  1. brazil_rosewood                (score: 0.0427)
  2. artemisia_herba_alba           (score: 0.0382)
  3. tobacco_flower                 (score: 0.0352)
  4. bulgarian_rose                 (score: 0.0290)
  5. lovage_root                    (score: 0.0284)
  6. pandanus_odoratissimus_oil     (score: 0.0205)
  7. italian_orange_oil             (score: 0.0205)
  8. thymus_caespeititius           (score: 0.0199)
  9. satureia_montana               (score: 0.0

## Case Study 2: Chocolate and Vanilla

In [48]:
print("="*80)
print("EXAMPLE 2: CHOCOLATE & VANILLA")
print("="*80)

# Analyze pair
result = analyze_ingredient_pair('cocoa', 'vanilla')

if 'error' not in result:
    print(f"\nAnalyzing: {result['name1'].upper()} <-> {result['name2'].upper()}\n")
    print(f"Basic Info:")
    print(f"  {result['name1']}: {result['category1']}, {result['n_compounds1']} compounds")
    print(f"  {result['name2']}: {result['category2']}, {result['n_compounds2']} compounds")
    print(f"\nChemical Similarity:")
    print(f"  Shared compounds: {result['n_shared']}")
    print(f"  Jaccard similarity: {result['jaccard_similarity']:.4f}")
    print(f"\nFlavor Network:")
    print(f"  Direct connection: {'YES' if result['connected_flavor'] else 'NO'}")
    if result['connected_flavor']:
        print(f"  Shared compounds (edge weight): {result['weight_flavor']}")
    print(f"  Path length: {result['path_length']}")
    if result['path_length'] < float('inf') and result['path_length'] <= 5:
        print(f"  Path: {' -> '.join(result['path_names'])}")
else:
    print(result['error'])

# Find bridges
print("\n" + "-"*80)
print("BRIDGE DISCOVERY")
print("-"*80)

print("\nMethod 1: Betweenness Centrality")
bridges_between = find_bridge_nodes('chocolate', 'vanilla', method='betweenness', top_n=10)
if bridges_between:
    for i, (name, score) in enumerate(bridges_between, 1):
        print(f"  {i}. {name:30s} (score: {score:.4f})")
else:
    print("  No bridges found")

print("\nMethod 2: Optimal Path Centrality")
bridges_path = find_bridge_nodes('chocolate', 'vanilla', method='optimal_path', top_n=10)
if bridges_path:
    for i, (name, score) in enumerate(bridges_path, 1):
        print(f"  {i}. {name:30s} (score: {score:.4f})")
else:
    print("  No bridges found")

print("\n" + "="*80)

EXAMPLE 2: CHOCOLATE & VANILLA

Analyzing: COCOA <-> VANILLA

Basic Info:
  cocoa: plant derivative, 193 compounds
  vanilla: spice, 78 compounds

Chemical Similarity:
  Shared compounds: 30
  Jaccard similarity: 0.1245

Flavor Network:
  Direct connection: YES
  Shared compounds (edge weight): 30
  Path length: 1
  Path: cocoa -> vanilla

--------------------------------------------------------------------------------
BRIDGE DISCOVERY
--------------------------------------------------------------------------------

Method 1: Betweenness Centrality
  No bridges found

Method 2: Optimal Path Centrality
  No bridges found



## Case Study 3: Salmon and Maple Syrup

In [49]:
print("="*80)
print("EXAMPLE 3: SALMON & MAPLE SYRUP")
print("="*80)

# Analyze pair
result = analyze_ingredient_pair('salmon', 'maple_syrup')

if 'error' not in result:
    print(f"\nAnalyzing: {result['name1'].upper()} <-> {result['name2'].upper()}\n")
    print(f"Basic Info:")
    print(f"  {result['name1']}: {result['category1']}, {result['n_compounds1']} compounds")
    print(f"  {result['name2']}: {result['category2']}, {result['n_compounds2']} compounds")
    print(f"\nChemical Similarity:")
    print(f"  Shared compounds: {result['n_shared']}")
    print(f"  Jaccard similarity: {result['jaccard_similarity']:.4f}")
    print(f"\nFlavor Network:")
    print(f"  Direct connection: {'YES' if result['connected_flavor'] else 'NO'}")
    if result['connected_flavor']:
        print(f"  Shared compounds (edge weight): {result['weight_flavor']}")
    print(f"  Path length: {result['path_length']}")
    if result['path_length'] < float('inf') and result['path_length'] <= 5:
        print(f"  Path: {' -> '.join(result['path_names'])}")
else:
    print(result['error'])

# Find bridges
print("\n" + "-"*80)
print("BRIDGE DISCOVERY")
print("-"*80)

print("\nMethod 1: Betweenness Centrality")
bridges_between = find_bridge_nodes('salmon', 'maple syrup', method='betweenness', top_n=10)
if bridges_between:
    for i, (name, score) in enumerate(bridges_between, 1):
        print(f"  {i}. {name:30s} (score: {score:.4f})")
else:
    print("  No bridges found")

print("\nMethod 2: Optimal Path Centrality")
bridges_path = find_bridge_nodes('salmon', 'maple syrup', method='optimal_path', top_n=10)
if bridges_path:
    for i, (name, score) in enumerate(bridges_path, 1):
        print(f"  {i}. {name:30s} (score: {score:.4f})")
else:
    print("  No bridges found")

print("\n" + "="*80)

EXAMPLE 3: SALMON & MAPLE SYRUP

Analyzing: SALMON <-> MAPLE_SYRUP

Basic Info:
  salmon: fish/seafood, 84 compounds
  maple_syrup: plant derivative, 3 compounds

Chemical Similarity:
  Shared compounds: 0
  Jaccard similarity: 0.0000

Flavor Network:
  Direct connection: NO
  Path length: 2
  Path: salmon -> port_wine -> maple_syrup

--------------------------------------------------------------------------------
BRIDGE DISCOVERY
--------------------------------------------------------------------------------

Method 1: Betweenness Centrality
  No bridges found

Method 2: Optimal Path Centrality
  No bridges found



## Interactive Analysis

This cell can be modified to explore custom ingredient pair combinations.

In [ ]:
# INTERACTIVE EXPLORATION
# 
# Example usage:
# 1. Search for ingredients:
#    matches = search_ingredients('chicken')
#    print(matches)
#
# 2. Analyze a pair:
#    result = analyze_ingredient_pair('chicken', 'lemon')
#    print(result)
#
# 3. Find bridges:
#    bridges = find_bridge_nodes('chocolate', 'coffee', method='betweenness')
#    for name, score in bridges:
#        print(f"{name}: {score:.4f}")

print("Analysis functions ready. Modify this cell to explore ingredient pairs.")